# Assignment B: Build a Reproducible Segmentation Experiment
**Course:** Advanced Computer Vision (CS107203CS) — Unit 3  

---
### Complete 8-Step Roadmap
1. **Choose a dataset:** Oxford-IIIT Pet (images of cats and dogs with segmentation annotations).
2. **Prepare the data:** Formal Data Contract (splits, resizing, normalization, mask remapping).
3. **Build the baseline:** Standard U-Net encoder-decoder architecture.
4. **Train the baseline:** Fixed seed, optimizer, learning rate, batch size, and stopping rule.
5. **Modify one mechanism:** Introduce Dilated (Atrous) Convolutions in the U-Net bottleneck.
6. **Train the modified model:** Keep the same dataset, training configuration, and computational budget.
7. **Evaluate both models:** Calculate per-class IoU, mIoU, parameter count, and inference latency.
8. **Compare the results:** Prediction images, failure cases, training curves, and critical discussion.

## 0. Environment Setup & Hardware Inspection

In [ ]:
import os
import sys
import time
import random
import json
from typing import Tuple, Dict, Any, List

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset
from torchvision import datasets, transforms
import torchvision.transforms.functional as TF
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# Set seeds for absolute reproducibility
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 1 & 2. Data Contract & Data Preparation (Oxford-IIIT Pet)

### **The Data Contract**
- **Dataset Choice:** Oxford-IIIT Pet Dataset (Parkhi et al., 2012).
- **Ground Truth:** Pixel-level Trimap annotations.
- **Class IDs & Remapping:**
  * Raw 1 $\rightarrow$ **0: Foreground (Pet)**
  * Raw 2 $\rightarrow$ **1: Background**
  * Raw 3 $\rightarrow$ **2: Boundary / Outline Contour**
- **Ignored Labels:** None (`ignore_index = -100`). All 3 classes are evaluated.
- **Splits:** 70% Train, 15% Validation, 15% Test (fixed random generator seed 42).
- **Resolution:** $256 \times 256$ pixels.
- **Transforms:**
  * Image: Bilinear resize, normalized by ImageNet mean `[0.485, 0.456, 0.406]` & std `[0.229, 0.224, 0.225]`.
  * Mask: Nearest-neighbor resize, remapped to `{0, 1, 2}`.
  * Augmentation (Train only): Synchronized Random Horizontal Flip ($p=0.5$).

In [ ]:
DATA_CONTRACT = {
    "dataset": "Oxford-IIIT Pet",
    "num_classes": 3,
    "class_names": ["Foreground (Pet)", "Background", "Boundary"],
    "class_ids": {0: "Foreground (Pet)", 1: "Background", 2: "Boundary"},
    "resolution": (256, 256),
    "normalization": {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
    "split_ratios": {"train": 0.70, "val": 0.15, "test": 0.15},
    "seed": 42
}

class SegmentationTransform:
    def __init__(self, size=(256, 256), is_train=True):
        self.size = size
        self.is_train = is_train
        self.normalize = transforms.Normalize(
            mean=DATA_CONTRACT["normalization"]["mean"],
            std=DATA_CONTRACT["normalization"]["std"]
        )

    def __call__(self, image: Image.Image, mask: Image.Image):
        image = TF.resize(image, self.size, interpolation=transforms.InterpolationMode.BILINEAR)
        mask = TF.resize(mask, self.size, interpolation=transforms.InterpolationMode.NEAREST)

        if self.is_train and random.random() > 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)

        image = TF.to_tensor(image)
        image = self.normalize(image)

        mask_np = np.array(mask, dtype=np.int64)
        remapped_mask = np.zeros_like(mask_np)
        remapped_mask[mask_np == 1] = 0  # Pet
        remapped_mask[mask_np == 2] = 1  # Background
        remapped_mask[mask_np == 3] = 2  # Boundary
        return image, torch.from_numpy(remapped_mask).long()

class OxfordPetWrapper(Dataset):
    def __init__(self, root="./data", split="trainval", is_train=True, download=True):
        self.dataset = datasets.OxfordIIITPet(root=root, split=split, target_types="segmentation", download=download)
        self.transform = SegmentationTransform(is_train=is_train)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, mask = self.dataset[idx]
        return self.transform(img.convert("RGB"), mask)

class SubsetTransform(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        actual_idx = self.subset.indices[idx]
        concat_ds = self.subset.dataset
        ds_idx = 0
        while actual_idx >= concat_ds.cumulative_sizes[ds_idx]:
            ds_idx += 1
        sample_idx = actual_idx if ds_idx == 0 else actual_idx - concat_ds.cumulative_sizes[ds_idx - 1]
        img, mask = concat_ds.datasets[ds_idx].dataset[sample_idx]
        return self.transform(img.convert("RGB"), mask)

print("Downloading / Loading Oxford-IIIT Pet Dataset...")
ds_tv = OxfordPetWrapper("./data", split="trainval", is_train=False, download=True)
ds_test = OxfordPetWrapper("./data", split="test", is_train=False, download=True)
combined = ConcatDataset([ds_tv, ds_test])
total = len(combined)

train_cnt = int(0.70 * total)
val_cnt = int(0.15 * total)
test_cnt = total - train_cnt - val_cnt

sub_train, sub_val, sub_test = random_split(combined, [train_cnt, val_cnt, test_cnt], generator=torch.Generator().manual_seed(SEED))

train_ds = SubsetTransform(sub_train, SegmentationTransform(is_train=True))
val_ds = SubsetTransform(sub_val, SegmentationTransform(is_train=False))
test_ds = SubsetTransform(sub_test, SegmentationTransform(is_train=False))

BATCH_SIZE = 16
NUM_WORKERS = 2 if sys.platform != 'win32' else 0
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Data Contract Verified!")
print(f"  Total samples: {total}")
print(f"  Train split:   {len(train_ds)} ({len(train_ds)/total*100:.1f}%)")
print(f"  Val split:     {len(val_ds)} ({len(val_ds)/total*100:.1f}%)")
print(f"  Test split:    {len(test_ds)} ({len(test_ds)/total*100:.1f}%)")

## 3. Architecture Definitions: Baseline vs Controlled Change (Dilated U-Net)

### **Scientific Hypothesis & Controlled Ablation**
- **Baseline:** Standard U-Net encoder-decoder with standard $3\times 3$ convolutions in the bottleneck ($dilation=1$).
- **Controlled Modification:** We replace the bottleneck standard convolutions with **Dilated (Atrous) Convolutions** with dilation rates $r_1=2$ and $r_2=4$.
- **Scientific Budget Parity:**
  - $\Delta$ Parameters $= 0$ (both use identical $3\times 3$ kernel weights and channel depths 512 $\rightarrow$ 512).
  - $\Delta$ Computational Footprint $\approx 0$.
  - **Isolated Variable:** Expansion of receptive field at the lowest spatial resolution without spatial downsampling.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, dilation=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=dilation, dilation=dilation, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=dilation, dilation=dilation, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.net(x)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x):
        return self.conv(self.pool(x))

class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size(2) - x1.size(2)
        diffX = x2.size(3) - x1.size(3)
        if diffX != 0 or diffY != 0:
            x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class BaselineUNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=3, base_c=32):
        super().__init__()
        c = base_c
        self.inc = DoubleConv(in_channels, c)
        self.down1 = DownBlock(c, c * 2)
        self.down2 = DownBlock(c * 2, c * 4)
        self.down3 = DownBlock(c * 4, c * 8)
        self.bottleneck = DoubleConv(c * 8, c * 16, dilation=1)  # standard dilation=1
        self.up1 = UpBlock(c * 16, c * 8)
        self.up2 = UpBlock(c * 8, c * 4)
        self.up3 = UpBlock(c * 4, c * 2)
        self.up4 = UpBlock(c * 2, c)
        self.outc = nn.Conv2d(c, num_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        b = self.bottleneck(x4)
        d1 = self.up1(b, x4)
        d2 = self.up2(d1, x3)
        d3 = self.up3(d2, x2)
        d4 = self.up4(d3, x1)
        return self.outc(d4)

class DilatedUNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=3, base_c=32):
        super().__init__()
        c = base_c
        self.inc = DoubleConv(in_channels, c)
        self.down1 = DownBlock(c, c * 2)
        self.down2 = DownBlock(c * 2, c * 4)
        self.down3 = DownBlock(c * 4, c * 8)
        
        # Controlled Change: Dilated Bottleneck (dilation rates 2 and 4)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(c * 8, c * 16, kernel_size=3, padding=2, dilation=2, bias=False),
            nn.BatchNorm2d(c * 16),
            nn.ReLU(inplace=True),
            nn.Conv2d(c * 16, c * 16, kernel_size=3, padding=4, dilation=4, bias=False),
            nn.BatchNorm2d(c * 16),
            nn.ReLU(inplace=True)
        )
        
        self.up1 = UpBlock(c * 16, c * 8)
        self.up2 = UpBlock(c * 8, c * 4)
        self.up3 = UpBlock(c * 4, c * 2)
        self.up4 = UpBlock(c * 2, c)
        self.outc = nn.Conv2d(c, num_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        b = self.bottleneck(x4)
        d1 = self.up1(b, x4)
        d2 = self.up2(d1, x3)
        d3 = self.up3(d2, x2)
        d4 = self.up4(d3, x1)
        return self.outc(d4)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

base_model = BaselineUNet(num_classes=3)
dila_model = DilatedUNet(num_classes=3)
print(f"Baseline U-Net Parameter Count: {count_params(base_model):,}")
print(f"Dilated U-Net Parameter Count:  {count_params(dila_model):,}")
print(f"Delta Parameters:               {count_params(dila_model) - count_params(base_model)} (Strict 100% Parameter Parity)")

## 4 & 6. Training Pipeline: Baseline vs Dilated U-Net

### **Controlled Training Budget**
- **Optimizer:** AdamW (`lr=1e-3`, `weight_decay=1e-4`)
- **Scheduler:** CosineAnnealingLR over 15 epochs
- **Loss Function:** Categorical Cross-Entropy Loss
- **Stopping Rule:** Fixed 15 epochs with best-checkpointing on Validation mIoU
- **Hardware Device:** GPU (CUDA) if available

In [ ]:
class MetricTracker:
    def __init__(self, num_classes=3, class_names=None):
        self.num_classes = num_classes
        self.class_names = class_names or ["Pet", "Background", "Boundary"]
        self.cm = np.zeros((num_classes, num_classes), dtype=np.int64)

    def reset(self):
        self.cm.fill(0)

    def update(self, preds, targets):
        if preds.dim() == 4:
            preds = torch.argmax(preds, dim=1)
        p = preds.view(-1).cpu().numpy()
        t = targets.view(-1).cpu().numpy()
        mask = (t >= 0) & (t < self.num_classes)
        hist, _, _ = np.histogram2d(t[mask], p[mask], bins=self.num_classes, range=[[0, self.num_classes], [0, self.num_classes]])
        self.cm += hist.astype(np.int64)

    def compute(self):
        tp = np.diag(self.cm)
        fp = np.sum(self.cm, axis=0) - tp
        fn = np.sum(self.cm, axis=1) - tp
        denom = tp + fp + fn
        ious = np.where(denom > 0, tp / denom, 0.0)
        return {
            "miou": float(np.mean(ious)),
            "per_class": {name: float(iou) for name, iou in zip(self.class_names, ious)},
            "pixel_acc": float(np.sum(tp) / (np.sum(self.cm) + 1e-12))
        }

def train_model(model, name, epochs=15, lr=1e-3):
    set_seed(SEED)
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    
    tracker = MetricTracker(3, DATA_CONTRACT["class_names"])
    history = {"epoch": [], "train_loss": [], "train_miou": [], "val_loss": [], "val_miou": []}
    best_val_miou = 0.0
    best_weights = None
    
    print(f"\n--- Starting Training: {name} ({epochs} epochs) ---")
    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        train_loss = 0.0
        tracker.reset()
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            outs = model(imgs)
            loss = criterion(outs, masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)
            tracker.update(outs.detach(), masks)
        
        t_loss = train_loss / len(train_loader.dataset)
        t_miou = tracker.compute()["miou"]
        scheduler.step()
        
        # Validation
        model.eval()
        val_loss = 0.0
        tracker.reset()
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                outs = model(imgs)
                loss = criterion(outs, masks)
                val_loss += loss.item() * imgs.size(0)
                tracker.update(outs, masks)
        
        v_loss = val_loss / len(val_loader.dataset)
        v_res = tracker.compute()
        v_miou = v_res["miou"]
        
        history["epoch"].append(epoch)
        history["train_loss"].append(round(t_loss, 4))
        history["train_miou"].append(round(t_miou, 4))
        history["val_loss"].append(round(v_loss, 4))
        history["val_miou"].append(round(v_miou, 4))
        
        if v_miou > best_val_miou:
            best_val_miou = v_miou
            best_weights = model.state_dict().copy()
            print(f"Epoch {epoch:02d}: Train Loss {t_loss:.4f}, mIoU {t_miou:.4f} | Val Loss {v_loss:.4f}, mIoU {v_miou:.4f} *BEST*")
        else:
            print(f"Epoch {epoch:02d}: Train Loss {t_loss:.4f}, mIoU {t_miou:.4f} | Val Loss {v_loss:.4f}, mIoU {v_miou:.4f}")
            
    model.load_state_dict(best_weights)
    return model, history

In [ ]:
# Execute Training
baseline_model, baseline_history = train_model(BaselineUNet(num_classes=3), name="Baseline U-Net", epochs=15)
dilated_model, dilated_history = train_model(DilatedUNet(num_classes=3), name="Dilated U-Net", epochs=15)

In [ ]:
# Save Best Checkpoints to Disk (.pth)
import os
os.makedirs('./checkpoints', exist_ok=True)
torch.save(baseline_model.state_dict(), './checkpoints/baseline_best_model.pth')
torch.save(dilated_model.state_dict(), './checkpoints/dilated_best_model.pth')
print('Checkpoints successfully saved to disk:')
print('  -> ./checkpoints/baseline_best_model.pth')
print('  -> ./checkpoints/dilated_best_model.pth')


## 7. Rigorous Evaluation: Evidence & Latency

### **Standardized Latency Conditions:**
- Single-image batch size ($1, 3, 256, 256$)
- 30 warm-up runs to stabilize GPU clocks / cache
- 100 timed runs with hardware synchronization
- Reported in milliseconds (Mean $\pm$ Std) and frames per second (FPS)

In [ ]:
def benchmark_latency(model, runs=100, warmup=30):
    model.eval()
    dummy = torch.randn(1, 3, 256, 256, device=device)
    is_cuda = (device.type == 'cuda')
    
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(dummy)
        if is_cuda: torch.cuda.synchronize()
        
        times = []
        if is_cuda:
            st, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
            for _ in range(runs):
                st.record()
                _ = model(dummy)
                end.record()
                torch.cuda.synchronize()
                times.append(st.elapsed_time(end))
        else:
            for _ in range(runs):
                t0 = time.perf_counter()
                _ = model(dummy)
                times.append((time.perf_counter() - t0) * 1000.0)
    return float(np.mean(times)), float(np.std(times))

base_lat_mean, base_lat_std = benchmark_latency(baseline_model)
dila_lat_mean, dila_lat_std = benchmark_latency(dilated_model)

# Evaluate on Test Set
def eval_test(model):
    model.eval()
    tr = MetricTracker(3, DATA_CONTRACT["class_names"])
    with torch.no_grad():
        for imgs, masks in test_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            tr.update(model(imgs), masks)
    return tr.compute()

base_test = eval_test(baseline_model)
dila_test = eval_test(dilated_model)

print("=" * 76)
print(f"{'Metric':<26} | {'Baseline U-Net':<22} | {'Dilated U-Net (Ours)':<22}")
print("-" * 76)
print(f"{'Trainable Parameters':<26} | {count_params(baseline_model):<22,d} | {count_params(dilated_model):<22,d}")
print(f"{'Latency (ms/img)':<26} | {base_lat_mean:.2f} ± {base_lat_std:.2f} ms     | {dila_lat_mean:.2f} ± {dila_lat_std:.2f} ms")
print(f"{'Throughput (FPS)':<26} | {1000.0/base_lat_mean:<22.1f} | {1000.0/dila_lat_mean:<22.1f}")
print(f"{'Test mIoU':<26} | {base_test['miou']:<22.4f} | {dila_test['miou']:<22.4f}")
print(f"{'Pixel Accuracy':<26} | {base_test['pixel_acc']:<22.4f} | {dila_test['pixel_acc']:<22.4f}")
print("-" * 76)
for c in DATA_CONTRACT["class_names"]:
    print(f"{c + ' IoU':<26} | {base_test['per_class'][c]:<22.4f} | {dila_test['per_class'][c]:<22.4f}")
print("=" * 76)

## 8. Results, Visual Comparisons, Failure Analysis & Training Curves

In [ ]:
# Visual Qualitative Comparison
palette = np.array([
    [220, 50, 50],    # Pet: Red/Coral
    [50, 160, 50],    # Background: Green
    [240, 200, 30]    # Boundary: Yellow/Amber
], dtype=np.uint8)

def colorize(m):
    rgb = np.zeros((*m.shape, 3), dtype=np.uint8)
    for i in range(3): rgb[m == i] = palette[i]
    return rgb

def denorm(t):
    mean, std = np.array(DATA_CONTRACT["normalization"]["mean"]), np.array(DATA_CONTRACT["normalization"]["std"])
    arr = t.permute(1, 2, 0).cpu().numpy() * std + mean
    return np.clip(arr * 255.0, 0, 255).astype(np.uint8)

baseline_model.eval()
dilated_model.eval()

imgs, gts, b_preds, d_preds, sample_ious = [], [], [], [], []
with torch.no_grad():
    for x, y in test_loader:
        xb = x.to(device)
        pb = torch.argmax(baseline_model(xb), dim=1).cpu().numpy()
        pd = torch.argmax(dilated_model(xb), dim=1).cpu().numpy()
        yb = y.numpy()
        for i in range(len(x)):
            imgs.append(x[i])
            gts.append(yb[i])
            b_preds.append(pb[i])
            d_preds.append(pd[i])
            inter = np.sum((pb[i] == yb[i]) & (yb[i] != 1))
            union = np.sum((pb[i] != 1) | (yb[i] != 1)) + 1e-6
            sample_ious.append(inter / union)
        if len(imgs) >= 40: break

# 1. Plot Success Comparisons
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
titles = ["Input Image", "Ground Truth", "Baseline U-Net", "Dilated U-Net"]
for j, t in enumerate(titles): axes[0, j].set_title(t, fontsize=13, fontweight='bold', pad=8)

for r, idx in enumerate([0, 2, 4, 6]):
    axes[r, 0].imshow(denorm(imgs[idx]))
    axes[r, 1].imshow(colorize(gts[idx]))
    axes[r, 2].imshow(colorize(b_preds[idx]))
    axes[r, 3].imshow(colorize(d_preds[idx]))
    for c in range(4): axes[r, c].axis('off')

plt.suptitle("Qualitative Predictions: Baseline vs. Dilated U-Net", fontsize=15, fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()

# 2. Plot Failure Cases
fail_indices = np.argsort(sample_ious)[:4]
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for j, t in enumerate(titles): axes[0, j].set_title(t, fontsize=13, fontweight='bold', pad=8)

for r, idx in enumerate(fail_indices):
    axes[r, 0].imshow(denorm(imgs[idx]))
    axes[r, 1].imshow(colorize(gts[idx]))
    axes[r, 2].imshow(colorize(b_preds[idx]))
    axes[r, 3].imshow(colorize(d_preds[idx]))
    for c in range(4): axes[r, c].axis('off')

plt.suptitle("Failure Cases Analysis (Challenging Poses & Blurred Fur Boundaries)", fontsize=15, fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()

# 3. Plot Training & Validation Curves
ep_range = baseline_history["epoch"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(ep_range, baseline_history["train_loss"], 'b--', label="Baseline Train")
ax1.plot(ep_range, baseline_history["val_loss"], 'b-', label="Baseline Val")
ax1.plot(ep_range, dilated_history["train_loss"], 'r--', label="Dilated Train")
ax1.plot(ep_range, dilated_history["val_loss"], 'r-', label="Dilated Val")
ax1.set_title("Cross-Entropy Loss Curve", fontweight='bold')
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, alpha=0.5)

ax2.plot(ep_range, baseline_history["train_miou"], 'b--', label="Baseline Train mIoU")
ax2.plot(ep_range, baseline_history["val_miou"], 'b-', label="Baseline Val mIoU")
ax2.plot(ep_range, dilated_history["train_miou"], 'r--', label="Dilated Train mIoU")
ax2.plot(ep_range, dilated_history["val_miou"], 'r-', label="Dilated Val mIoU")
ax2.set_title("Validation Mean IoU Curve", fontweight='bold')
ax2.set_xlabel("Epoch")
ax2.set_ylabel("mIoU")
ax2.legend()
ax2.grid(True, alpha=0.5)

plt.tight_layout()
plt.show()

## 9. Critical Scientific Discussion: What the Experiment Does NOT Establish

As required by the assignment rubric, rigorous experimental science requires documenting not only what the data demonstrates, but also **what the experiment fails to establish**:

1. **Generalization across Out-of-Distribution Domains:**
   - This experiment tests pet segmentation in domestic, human-centric photographs (Oxford-IIIT Pet). It **does not establish** that dilated convolutions improve segmentation in autonomous driving (e.g. Cityscapes), satellite remote sensing, or endoscopic medical imagery where texture distributions and object scale variances differ radically.

2. **Superiority Across Alternative Resolutions:**
   - All experiments were trained strictly at $256 \times 256$. At higher resolutions (e.g., $1024 \times 1024$ or $2048 \times 1024$), the optimal dilation rate shifts significantly. A dilation rate of $r=2, 4$ that provides sufficient receptive field at $256 \times 256$ may be negligible at $1024 \times 1024$.

3. **Invariance to Grid/Gridding Artifacts:**
   - Dilated convolutions are mathematically known to introduce the *"gridding effect"* (loss of local information due to sparse sampling). While global context is expanded, fine high-frequency hair/fur details might suffer without hybrid dense-dilated filtering.

4. **Algorithmic Convergence with Different Pretrained Backbones:**
   - The models were trained from scratch with standard channel initialization. This experiment **does not establish** how dilated bottleneck modifications interact with pre-trained backbones (e.g., ResNet-50, Swin Transformer, or ConvNeXt pre-trained on ImageNet-1k).

---
### References
1. Ronneberger, O., Fischer, P., & Brox, T. (2015). *U-Net: Convolutional Networks for Biomedical Image Segmentation*. MICCAI.
2. Chen, L. C., Papandreou, G., Kokkinos, I., Murphy, K., & Yuille, A. L. (2017). *DeepLab: Semantic Image Segmentation with Deep Convolutional Nets, Atrous Convolution, and Fully Connected CRFs*. IEEE TPAMI.
3. Parkhi, O. M., Vedaldi, A., Zisserman, A., & Jawahar, C. V. (2012). *Cats and Dogs*. IEEE CVPR.